# Weights & Biases demo: tracking, sweeps and artifacts

A small CNN on the **MNIST** dataset (28×28 handwritten-digit images).
The dataset is downloaded on first use; training time depends on your hardware.

**What this notebook shows**
1. **Experiment tracking**: `wandb.init`, config, `run.log` → live charts
2. **Sweeps**: compare different training settings
3. **Artifacts**: the trained model is versioned and loaded back by alias

In [6]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.datasets import MNIST
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
RUNS_DIR = os.path.abspath("runs")
os.makedirs(RUNS_DIR, exist_ok=True)
os.environ["WANDB_DIR"] = RUNS_DIR

import wandb


wandb.login()   # paste your API key from https://wandb.ai/authorize the first time

wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from /home/neo/.netrc.
wandb: Currently logged in as: eminacar (eminacar-itu) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


True

## 1. Data and model

In [7]:
train_data = MNIST(root="./data", train=True, download=True)
test_data = MNIST(root="./data", train=False, download=True)
X = train_data.data.numpy().astype(np.float32)[:, None] / 255.0
y = train_data.targets.numpy()
X_te = test_data.data.numpy().astype(np.float32)[:, None] / 255.0
y_te = test_data.targets.numpy()
# Use half of the training data; keep the official test set unchanged.
X, _, y, _ = train_test_split(X, y, train_size=0.5, stratify=y, random_state=42)
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

def loader(X, y, bs, shuffle=False):
    return DataLoader(TensorDataset(torch.from_numpy(X), torch.from_numpy(y)), batch_size=bs, shuffle=shuffle)

print(f"train {len(y_tr)} | val {len(y_va)} | test {len(y_te)}")


train 24000 | val 6000 | test 10000


In [8]:
class SmallCNN(nn.Module):
    def __init__(self, channels=8, dropout=0.2):
        super().__init__()
        self.conv1 = nn.Conv2d(1, channels, 3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels * 2, 3, padding=1)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(channels * 2 * 14 * 14, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)   # 28x28 -> 14x14
        return self.fc(self.drop(x.flatten(1)))

@torch.no_grad()
def evaluate(model, dl):
    model.eval()
    losses, preds, probs = [], [], []
    for xb, yb in dl:
        out = model(xb)
        losses.append(F.cross_entropy(out, yb, reduction="sum").item())
        probs.append(out.softmax(1))
    probs = torch.cat(probs)
    preds = probs.argmax(1).numpy()
    labels = dl.dataset.tensors[1].numpy()
    return sum(losses) / len(labels), (preds == labels).mean(), preds, probs.numpy()

## 2. Training with W&B tracking

Start a run, log training loss and gradient norm each step and validation accuracy each epoch, and save the model as an artifact.
The learning rate stays fixed during each run. Test accuracy is printed here only, as a final check on unseen data.

In [ ]:
DEFAULTS = dict(epochs=7, batch_size=64, lr=1e-3, channels=8, dropout=0.2, seed=42)
PROJECT = "wandb-demo2"

def run_name(cfg):
    from datetime import datetime
    date = datetime.now().strftime("%d%m")
    return f"lr{cfg.lr:.3g}-b{cfg.batch_size}-c{cfg.channels}-d{cfg.dropout:.2f}-{date}"

def train(config=DEFAULTS):
    # 1. Start a run and record the settings
    with wandb.init(project=PROJECT, config=config, settings=wandb.Settings(quiet=True)) as run:
        cfg = run.config
        run.name = run_name(cfg)
        torch.manual_seed(cfg.seed)
        train_dl = loader(X_tr, y_tr, cfg.batch_size, shuffle=True)
        val_dl = loader(X_va, y_va, 256)
        test_dl = loader(X_te, y_te, 256)
        model = SmallCNN(cfg.channels, cfg.dropout)
        optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)

        # 2. Log training metrics at each batch step
        run.define_metric("train/*", step_metric="step")
        run.define_metric("val/*", step_metric="step")
        step = 0
        for epoch in range(cfg.epochs):
            model.train()
            for images, labels in train_dl:
                optimizer.zero_grad()
                loss = F.cross_entropy(model(images), labels)
                loss.backward()
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), float("inf"))
                optimizer.step()
                step += 1
                run.log({"step": step, "train/loss": loss.item(),
                         "train/grad_norm": grad_norm.item()})

            val_loss, val_acc, _, _ = evaluate(model, val_dl)
            run.log({"step": step, "epoch": epoch + 1, "val/loss": val_loss, "val/acc": val_acc})

        # Final check on unseen data (printed only, not logged to W&B)
        _, test_acc, _, _ = evaluate(model, test_dl)

        # 3. Save the model so it can be downloaded later
        run_dir = os.path.join(RUNS_DIR, run.id)
        os.makedirs(run_dir, exist_ok=True)
        model_path = os.path.join(run_dir, "model.pt")
        torch.save(model.state_dict(), model_path)
        artifact = wandb.Artifact("mnist-cnn", type="model", metadata=dict(cfg))
        artifact.add_file(model_path, name="model.pt")
        run.log_artifact(artifact)

    print(f"Training complete | Test accuracy: {test_acc:.1%}")
    print(f"W&B run: {run.url}")


## 3. One run
Click the run link that appears to watch the charts update live.

In [10]:
train()

Training complete | Test accuracy: 98.2%
W&B run: https://forge.coreweave.com/wandb/eminacar-itu/wandb-demo/runs/lqtmqckc


In [ ]:
# A second run with a different config, to compare the two in the workspace
train({**DEFAULTS, "lr": 0.005, "epochs": 7})

Training complete | Test accuracy: 98.0%
W&B run: https://forge.coreweave.com/wandb/eminacar-itu/wandb-demo/runs/lflbgan5


## 4. Hyperparameter sweep

A separate thing from section 2. The W&B server picks settings (grid search here), and an *agent* runs one small training function for each proposal.
That function is `sweep_run()` below. It logs training metrics each step and `val/acc` each epoch; the sweep optimizes `val/acc`. Each run uses a fixed learning rate from the list below.

In [12]:
def sweep_run():
    with wandb.init() as run:               # no config here: the sweep agent fills run.config
        cfg = run.config
        run.name = run_name(cfg)
        torch.manual_seed(DEFAULTS["seed"])
        tr, va = loader(X_tr, y_tr, cfg.batch_size, True), loader(X_va, y_va, 256)

        model = SmallCNN(cfg.channels, cfg.dropout)
        opt = torch.optim.Adam(model.parameters(), lr=cfg.lr)

        run.define_metric("train/*", step_metric="step")
        run.define_metric("val/*", step_metric="step")
        step = 0
        for epoch in range(cfg.epochs):
            model.train()
            for xb, yb in tr:
                opt.zero_grad()
                loss = F.cross_entropy(model(xb), yb)
                loss.backward()
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), float("inf"))
                opt.step()
                step += 1
                run.log({"step": step, "train/loss": loss.item(),
                         "train/grad_norm": grad_norm.item()})
            va_loss, va_acc, _, _ = evaluate(model, va)
            run.log({"step": step, "epoch": epoch + 1, "val/loss": va_loss, "val/acc": va_acc})

In [13]:
sweep_config = {
    "method": "grid",
    "metric": {"name": "val/acc", "goal": "maximize"},
    "parameters": {
        "lr":         {"values": [0.001, 0.01]},
        "batch_size": {"value": 64},
        "channels":   {"values": [8, 16]},
        "dropout":    {"values": [0.1, 0.3]},
        "epochs":     {"value": 4},
    },
}
sweep_id = wandb.sweep(sweep_config, project=PROJECT)   # register the search on the server


Create sweep with ID: hz3wgux4
Sweep URL: https://forge.coreweave.com/wandb/eminacar-itu/wandb-demo/sweeps/hz3wgux4


In [14]:
from IPython.utils.io import capture_output

with capture_output() as sweep_logs:
    wandb.agent(sweep_id, function=sweep_run, count=10)

print(f"Sweep finished: {sweep_id}")
# To inspect detailed logs: sweep_logs.show()


Sweep finished: hz3wgux4


## 5. Load the model back from Artifacts

Ask for the model by **alias** (`latest`, or a version like `v3`), not by a file path.
This run is recorded as a consumer, so the artifact's *Lineage* tab shows `train run → mnist-cnn → this run`.

In [15]:
with wandb.init(project=PROJECT, job_type="evaluate", settings=wandb.Settings(quiet=True)) as run:
    from datetime import datetime
    run.name = f"eval-{datetime.now():%d%m}"
    art = run.use_artifact("mnist-cnn:latest")
    cfg = art.metadata
    model = SmallCNN(cfg["channels"], cfg["dropout"])
    model.load_state_dict(torch.load(os.path.join(art.download(root=os.path.join(RUNS_DIR, "artifacts", art.name)), "model.pt")))
    _, acc, _, _ = evaluate(model, loader(X_te, y_te, 256))

print(f"Loaded model: {art.name} | Test accuracy: {acc:.1%}")

wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from /home/neo/.netrc.


wandb:   1 of 1 files downloaded.  


Loaded model: mnist-cnn:latest | Test accuracy: 98.0%
